# Explanation Notebook for ScenarioModel10
*Model Type:* **DecisionTree**
*User:* UserJSON (Expertise: Intermediate, Formats: [plainText], Details: medium, Purpose: learning)
*Explanation Method:* PDP *(auto-selected)*


## Training the DecisionTree Model
We train a **DecisionTree** model on the provided dataset.


In [1]:
import warnings; warnings.filterwarnings('ignore')
!pip install -q scikit-learn pandas matplotlib
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_json(r"data/breast_cancer.json")
y = df['target']
X = df.drop(columns=['target']).values
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
    print('[Info] Used stratified split to preserve class proportions in train/test.')
except Exception:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
    print('[Info] Used non-stratified split (stratification not applicable).')
class Dummy: pass
data = Dummy(); data.feature_names = list(df.drop(columns=['target']).columns)
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=0)
print('[Info] Normalization disabled by training policy.')
model.fit(X_train, y_train)
print('Accuracy: DecisionTree = ' + format(model.score(X_test, y_test), '.2f'))


[Info] Used stratified split to preserve class proportions in train/test.
[Info] Normalization disabled by training policy.
Accuracy: DecisionTree = 0.94


## Explaining the model using PDP
Partial Dependence Plot (PDP) shows how a feature affects the model's predictions on average.


In [2]:
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt

def _get_feature_names():
    try:
        names = list(getattr(data, 'feature_names', []))
        if names and len(names) == X_train.shape[1]:
            return names
    except Exception:
        pass
    if 'df' in globals():
        cols = [c for c in df.columns if c.lower() not in ('label','target','y')]
        if len(cols) == X_train.shape[1]:
            return cols
    return [f'f{j}' for j in range(X_train.shape[1])]

RAW_NAMES = _get_feature_names()
def _humanize(name):
    s = name.replace('_',' ').strip()
    s = re.sub(r'(?<!^)(?=[A-Z])',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return ' '.join([w.upper() if w.lower() in {'svm','pdp','ice','api','url'} else w.capitalize() for w in s.split(' ')])
HUMAN = [_humanize(n) for n in RAW_NAMES]

max_display = 6
table_rows = 10
curve_resolution = 24
ice_instances = 10
metric_digits = 4
figure_height = 4.6
line_width = 2.0
pdp_sample_rows = 8
ice_detailed_instances = 3
sampled_curve_points = 7
print('\n[PDP overview]')
print('Partial dependence shows the average effect of a feature on the model prediction.')
feat_idx = 0
xs = X_train[:, feat_idx]
grid = np.linspace(np.min(xs), np.max(xs), num=curve_resolution)
preds = []
for g in grid:
    X_tmp = X_train.copy()
    X_tmp[:, feat_idx] = g
    if hasattr(model, 'predict_proba'):
        preds.append(float(np.mean(model.predict_proba(X_tmp)[:,1])))
    else:
        preds.append(float(np.mean(model.predict(X_tmp))))
df_pdp = pd.DataFrame({'Feature':[HUMAN[feat_idx]]*len(grid), 'Value':grid, 'AveragePrediction':preds})
start_pred = float(df_pdp['AveragePrediction'].iloc[0])
end_pred = float(df_pdp['AveragePrediction'].iloc[-1])
delta_pred = end_pred - start_pred
if delta_pred > 0.01:
    trend_label = 'overall increasing'
elif delta_pred < -0.01:
    trend_label = 'overall decreasing'
else:
    trend_label = 'mostly stable'
sample_idx = np.unique(np.linspace(0, len(df_pdp)-1, num=min(pdp_sample_rows, len(df_pdp))).astype(int))
df_pdp_view = df_pdp.iloc[sample_idx].copy()
df_pdp_view['DeltaFromStart'] = df_pdp_view['AveragePrediction'] - start_pred
print('\n[Plain-text explanation]')
print("This explanation is presented in a learning-oriented style to help understand the model globally.")
print('Feature under analysis: ' + HUMAN[feat_idx])
print('How to read this explanation: each point summarizes the average prediction obtained when the selected feature is forced to a given value across the data.')
print('Overall trend: ' + trend_label)
print('Start prediction: ' + format(start_pred, '.4f'))
print('End prediction:   ' + format(end_pred, '.4f'))



[PDP overview]
Partial dependence shows the average effect of a feature on the model prediction.

[Plain-text explanation]
This explanation is presented in a learning-oriented style to help understand the model globally.
Feature under analysis: Mean Radius
How to read this explanation: each point summarizes the average prediction obtained when the selected feature is forced to a given value across the data.
Overall trend: mostly stable
Start prediction: 0.6286
End prediction:   0.6220
